# Exercice 3 - Modèle de Merton

On veut retrouver la valeur des actifs $V_t$ et leur volatilité $\sigma^V$ à partir du prix de l'equity et de sa volatilité observée.

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve

## Question 1 - Démonstration

### Obtention du systeme

Dans le modele de Merton, les actifs suivent un GBM et la dette est un zero-coupon de maturité T.
L'equity c'est un call européen sur V avec strike D :

$$S_t = V_t N(d_1) - D e^{-rT} N(d_2)$$

C'est juste Black-Scholes. Ca nous donne la premiere equation.

Pour la deuxieme, on applique Itô à $S = f(V,t)$. La partie diffusion donne :
$$\sigma^S S_t dW = \frac{\partial S}{\partial V} \sigma^V V_t dW$$

Or le delta du call c'est $\frac{\partial S}{\partial V} = N(d_1)$, donc :
$$\sigma^S = \sigma^V \frac{V_t}{S_t} N(d_1)$$

### Unicité

La premiere équation est croissante en V (call croissant en sous-jacent) et croissante en $\sigma^V$ (vega > 0). La deuxieme donne une relation monotone entre V et $\sigma^V$ pour $\sigma^S$ fixé. Le systeme a donc une unique solution.

## Question 2 - Résolution numérique

In [ ]:
# données
S = 25
D = 98.78
r = 0.05
T = 1
sig_S = 0.30

In [ ]:
def systeme(x):
    V, sig_V = x
    
    d1 = (np.log(V/D) + (r + 0.5*sig_V**2)*T) / (sig_V * np.sqrt(T))
    d2 = d1 - sig_V * np.sqrt(T)
    
    # eq 1 : S = V*N(d1) - D*exp(-rT)*N(d2)
    f1 = V * norm.cdf(d1) - D * np.exp(-r*T) * norm.cdf(d2) - S
    
    # eq 2 : sig_S = sig_V * V/S * N(d1)
    f2 = sig_V * V/S * norm.cdf(d1) - sig_S
    
    return [f1, f2]

In [ ]:
# point de depart
V0 = S + D * np.exp(-r*T)
sig0 = sig_S * S / V0

sol = fsolve(systeme, [V0, sig0])
V_star, sig_V_star = sol

print(f"V_t = {V_star:.2f}")
print(f"sigma_V = {sig_V_star:.4f} = {sig_V_star*100:.2f}%")

In [ ]:
# verification
d1 = (np.log(V_star/D) + (r + 0.5*sig_V_star**2)*T) / (sig_V_star*np.sqrt(T))
d2 = d1 - sig_V_star*np.sqrt(T)

S_verif = V_star * norm.cdf(d1) - D*np.exp(-r*T)*norm.cdf(d2)
sig_verif = sig_V_star * V_star/S * norm.cdf(d1)

print(f"Verif S = {S_verif:.4f} (attendu 25)")
print(f"Verif sigma_S = {sig_verif:.4f} (attendu 0.30)")
print(f"\nProbabilité de défaut (RN) = N(-d2) = {norm.cdf(-d2)*100:.2f}%")
print(f"Distance to default = d2 = {d2:.2f}")

### Interprétation

La valeur implicite des actifs (~119) est bien supérieure à l'equity (25), ce qui est normal puisque V = equity + dette. La volatilité des actifs (~6%) est beaucoup plus faible que celle de l'equity (30%), c'est l'effet de levier de Merton : la dette amplifie la vol des actions.

La PD risk-neutral est très faible car V est largement au dessus de D (levier modéré).